# Korea KNOC Dashboard

KNOC/MOTIE data from `scripts/update_korea.py` →
`data/processed/korea/korea_knoc.parquet` (consumption + closing stocks).

## Sections
1. **Setup** — load parquet; demand kbpm → kbd
2. **Headline demand** — end-use total (excl. naphtha)
3. **Native products**
4. **Canonical rollup**
5. **Recent trends**
6. **Seasonality index** — monthly index, last 5 complete years
7. **Seasonality by year** — native or canonical toggle
8. **KNOC vs JODI**
9. **Jet fuel vs Kayrros**
10. **Closing stocks** — levels and changes since late Feb 2026
11. **KNOC vs JODI (stocks)** — Petronet closing stocks vs JODI `CLOSTLV`

## Conventions
- Demand native unit **kbpm**; stocks native unit **kb** (thousand barrels at month-end).
- `product_native` is normalized English snake_case (`reference/korea.py`).
- Headline totals exclude **naphtha** (petchem-heavy); naphtha appears in **seasonality** and **JODI** (``NAPHTHA``). Fuel-oil JODI panel sums four KNOC rows vs ``RESFUEL``.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "update_korea.py").exists():
            return candidate
        if (candidate / "country_oil_scraper" / "scripts" / "update_korea.py").exists():
            return candidate / "country_oil_scraper"
    raise RuntimeError(f"Could not locate project root from cwd: {here}")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics.products import CANONICAL_KIND_LABEL, SUBCATEGORY_TO_PRODUCT_KIND
from analytics.units import convert_series
from reference.korea import (
    CHART_PRODUCTS,
    DELIVERY_HEADLINE_NATIVE,
    DISPLAY_LABELS,
    JODI_COMPARE_SERIES,
    KNOC_STOCKS_METRIC_TYPE,
    KNOC_UNIT_NATIVE,
    UNITS_KIND,
    knoc_series_for_jodi,
)

PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "korea" / "korea_knoc.parquet"

df = pd.read_parquet(PARQUET_PATH)
df["date"] = pd.to_datetime(df["date"])
demand = df[df["metric_type"] == "TOTDEMO"].copy()
stocks = df[df["metric_type"] == KNOC_STOCKS_METRIC_TYPE].copy()
demand["product_kind"] = demand["product_native"].map(UNITS_KIND)
demand["value_kbd"] = convert_series(
    demand["value"],
    KNOC_UNIT_NATIVE,
    "kbd",
    product_kind=demand["product_kind"],
    date=demand["date"],
)
stocks["value_mbbl"] = stocks["value"] / 1000.0

headline = demand[demand["product_native"].isin(DELIVERY_HEADLINE_NATIVE)].copy()
demand_canonical = (
    demand[demand["product_canonical"].notna()]
    .groupby(["date", "product_canonical", "is_provisional"], as_index=False)["value_kbd"]
    .sum()
)
demand_canonical["panel"] = demand_canonical["product_canonical"].map(
    lambda s: CANONICAL_KIND_LABEL.get(SUBCATEGORY_TO_PRODUCT_KIND.get(s, ""), s)
)

print(f"Loaded: {len(df):,} rows  ({df['date'].min().date()} -> {df['date'].max().date()})")
print(f"  demand rows: {len(demand):,}  |  stocks rows: {len(stocks):,}")
print(f"Headline natives: {len(DELIVERY_HEADLINE_NATIVE)}")
_s = demand[(demand.product_native == "gasoline") & (demand.date == "2024-01-01")]
if len(_s):
    print(f"  Gasoline Jan-2024: {_s.value_kbd.iloc[0]:,.1f} kbd (JODI ~259)")


Loaded: 8,333 rows  (1992-01-01 -> 2026-05-01)
  demand rows: 4,891  |  stocks rows: 3,442
Headline natives: 9
  Gasoline Jan-2024: 258.3 kbd (JODI ~259)


## 2. Headline total (excl. naphtha)


In [2]:
total = headline.groupby(["date", "is_provisional"], as_index=False)["value_kbd"].sum()
fig = px.line(total, x="date", y="value_kbd", title="Korea petroleum demand — headline (kbd)")
roll = total["value_kbd"].rolling(12, min_periods=6).mean()
fig.add_scatter(x=total["date"], y=roll, mode="lines", name="12m rolling avg", line=dict(dash="dot"))
fig.update_layout(height=440, template="plotly_white", hovermode="x unified")
fig.show()


## 3. Native headline products


In [3]:
mp = headline[headline["product_native"].isin(CHART_PRODUCTS)].copy()
mp["label"] = mp["product_native"].map(DISPLAY_LABELS)
fig = px.line(mp, x="date", y="value_kbd", color="label", title="KNOC consumption by product (kbd)")
fig.update_layout(height=480, hovermode="x unified", template="plotly_white")
fig.show()


## 4. Canonical products


In [4]:
fig = px.line(
    demand_canonical, x="date", y="value_kbd", color="panel",
    title="Canonical products (kbd)",
)
fig.update_layout(height=480, template="plotly_white", hovermode="x unified")
fig.show()


## 5. Recent trends (last 24 months)


In [5]:
cutoff = headline["date"].max() - pd.DateOffset(months=23)
recent = headline[(headline["date"] >= cutoff) & headline["product_native"].isin(CHART_PRODUCTS)].copy()
recent["label"] = recent["product_native"].map(DISPLAY_LABELS)
fig = px.line(recent, x="date", y="value_kbd", color="label", title="Last 24 months (kbd)")
fig.update_layout(height=420, template="plotly_white")
fig.show()
tbl = recent.sort_values(["label", "date"]).copy()
tbl["mom_pct"] = tbl.groupby("label")["value_kbd"].pct_change() * 100
tbl["yoy_pct"] = tbl.groupby("label")["value_kbd"].pct_change(12) * 100
display(tbl.groupby("label").tail(1)[["date", "value_kbd", "mom_pct", "yoy_pct"]].round(1))


,date,value_kbd,mom_pct,yoy_pct
8322,2026-05-01,330.2,-0.2,-6.0
8320,2026-05-01,6.7,-20.1,-82.7
8321,2026-05-01,4.2,1.4,-10.9
8324,2026-05-01,0.0,-3.2,78.6
8327,2026-05-01,1.1,-48.5,-59.8
8323,2026-05-01,250.2,8.9,6.4
8325,2026-05-01,7.5,-50.9,-92.8
8326,2026-05-01,13.7,-21.4,-22.4
8328,2026-05-01,268.7,-7.3,-18.6


## 6. Seasonality index (last 5 complete years)

Monthly demand indexed to 100 = annual mean, collapsed over the last five **complete** calendar years. Includes **naphtha** (same product set as the native seasonality-by-year view). Use this to see the typical within-year shape; Section 7 shows how each calendar year compares.


In [6]:
from reference.korea import SEASONALITY_NATIVE_PRODUCTS

last_year = int(demand.loc[~demand["is_provisional"], "date"].dt.year.max())
years = list(range(last_year - 5, last_year))
mp_idx = demand[demand["product_native"].isin(SEASONALITY_NATIVE_PRODUCTS)].copy()
mp_idx["label"] = mp_idx["product_native"].map(DISPLAY_LABELS)
mp_idx["year"] = mp_idx["date"].dt.year
mp_idx["month"] = mp_idx["date"].dt.month
mp_idx = mp_idx[mp_idx["year"].isin(years)]

annual = mp_idx.groupby(["label", "year"])["value_kbd"].mean().rename("annual_mean")
monthly = mp_idx.groupby(["label", "year", "month"])["value_kbd"].mean().reset_index()
monthly = monthly.merge(annual, on=["label", "year"])
monthly["index"] = 100 * monthly["value_kbd"] / monthly["annual_mean"]

fig = px.line(
    monthly,
    x="month",
    y="index",
    color="label",
    facet_col="label",
    facet_col_wrap=3,
    title=f"Seasonality index (100 = annual mean, {years[0]}–{years[-1]})",
)
fig.update_xaxes(tickmode="linear", tick0=1, dtick=1)
fig.update_layout(height=1200, template="plotly_white", showlegend=False)
fig.show()


## 7. Seasonality by calendar year

Use the dropdown to switch **native** (KNOC reporting rows, including **naphtha** and four fuel-oil splits) vs **canonical** (rollup via `product_map`, one **Fuel oil** panel plus **Naphtha**). Under the hood this only changes which DataFrame and `product_col` are passed to `seasonality_by_year_chart`.


In [7]:
import ipywidgets as widgets
from analytics import seasonality_by_year_chart
from reference.korea import seasonality_chart_inputs

# Default view: canonical (single Fuel oil panel). Change dropdown to re-run.
DEFAULT_SEASONALITY_VIEW = "canonical"


def plot_seasonality(view: str = DEFAULT_SEASONALITY_VIEW) -> None:
    season_df, product_col, products, labels, suffix = seasonality_chart_inputs(
        view,
        demand=demand,
        demand_canonical=demand_canonical,
    )
    if season_df.empty:
        print(f"No data for view={view!r}")
        return
    fig = seasonality_by_year_chart(
        season_df,
        products=products,
        product_col=product_col,
        value_col="value_kbd",
        product_labels=labels,
        highlight_year=int(season_df["date"].dt.year.max()),
        default_visible_prior_years=5,
        title=f"Seasonality by calendar year — Korea KNOC ({suffix}, kbd)",
        units_label="kbd",
    )
    fig.show()
    print(f"View: {view} — {len(products)} panels: {products}")


view_picker = widgets.Dropdown(
    options=[("Canonical (rollup)", "canonical"), ("Native (reporting rows)", "native")],
    value=DEFAULT_SEASONALITY_VIEW,
    description="Product view",
)
widgets.interact(plot_seasonality, view=view_picker)


interactive(children=(Dropdown(description='Product view', options=(('Canonical (rollup)', 'canonical'), ('Nat…

<function __main__.plot_seasonality(view: str = 'canonical') -> None>

## 8. KNOC vs JODI (TOTDEMO, kbd)

Small-multiples grid (same layout as Thailand EPPO vs JODI): **KNOC (Korea)** in blue, **JODI** in orange, one panel per product. Fuel-oil panel sums the four KNOC fuel-oil components vs JODI `RESFUEL`.


In [8]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run python scripts/update_jodi.py first.")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    knoc_parts = []
    jodi_codes = []
    for key, spec in JODI_COMPARE_SERIES.items():
        kn = knoc_series_for_jodi(demand, key, value_col="value_kbd")
        if kn.empty:
            continue
        kn = kn.groupby("date", as_index=False)["value_kbd"].sum()
        kn["panel"] = spec.panel
        knoc_parts.append(kn[["date", "panel", "value_kbd"]])
        jodi_codes.append(spec.jodi_energy_product)

    knoc_panel = pd.concat(knoc_parts, ignore_index=True)

    jodi_kr = jodi[
        (jodi["ref_area"] == "KR")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
        & (jodi["energy_product"].isin(jodi_codes))
    ].copy()
    code_to_panel = {
        spec.jodi_energy_product: spec.panel for spec in JODI_COMPARE_SERIES.values()
    }
    jodi_kr["panel"] = jodi_kr["energy_product"].map(code_to_panel)
    jodi_kr["value_kbd"] = jodi_kr["obs_value"]

    # Panel order aligned with Thailand dashboard (Diesel, Fuel oil, Gasoline, …).
    _order = ["Diesel", "Fuel oil", "Gasoline", "Kerosene", "Jet fuel", "LPG"]
    panels = [p for p in _order if p in knoc_panel["panel"].unique()]

    fig = cross_source_comparison_chart(
        df_a=knoc_panel,
        df_b=jodi_kr,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kbd",
        value_col_b="value_kbd",
        label_a="KNOC (Korea)",
        label_b="JODI",
        title="Korea TOTDEMO — KNOC vs JODI (kbd)",
        units_label="kbd",
        cols=2,
        panel_height=280,
    )
    fig.show()

    cutoff_24 = knoc_panel["date"].max() - pd.DateOffset(months=23)
    print("\nMean |gap| over last 24 months (kbd):")
    for panel in panels:
        k = knoc_panel.loc[knoc_panel["panel"] == panel].set_index("date")["value_kbd"]
        j = jodi_kr.loc[jodi_kr["panel"] == panel].set_index("date")["value_kbd"]
        merged = pd.concat([k, j], axis=1, keys=["knoc", "jodi"]).dropna()
        merged = merged.loc[merged.index >= cutoff_24]
        if merged.empty:
            continue
        gap = (merged["knoc"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:12s}  mean|gap| = {gap:>8,.1f} kbd  ({pct:5.1f}% of JODI level)")



Mean |gap| over last 24 months (kbd):
  Diesel        mean|gap| =     37.3 kbd  (  8.5% of JODI level)
  Fuel oil      mean|gap| =    135.9 kbd  ( 74.3% of JODI level)
  Gasoline      mean|gap| =      5.5 kbd  (  2.1% of JODI level)
  Kerosene      mean|gap| =      3.7 kbd  (  6.5% of JODI level)
  Jet fuel      mean|gap| =     63.6 kbd  ( 40.2% of JODI level)
  LPG           mean|gap| =      5.4 kbd  (  1.5% of JODI level)


In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if not JODI_PARQUET.exists():
    print("[skip] JODI parquet missing")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])
    jodi_kr = jodi[
        (jodi["ref_area"] == "KR")
        & (jodi["flow_breakdown"] == "TOTDEMO")
        & (jodi["unit_measure"] == "KBD")
    ]

    # --- KNOC totals ---
    knoc_headline_total = (
        headline.groupby("date", as_index=False)["value_kbd"].sum()
        .rename(columns={"value_kbd": "knoc_kbd"})
    )
    knoc_all_total = (
        demand[demand["product_native"] != "total"]
        .groupby("date", as_index=False)["value_kbd"]
        .sum()
        .rename(columns={"value_kbd": "knoc_all_kbd"})
    )

    # KNOC sum of §7-compare products (same scope as product panels)
    knoc_cmp_parts = []
    for key in JODI_COMPARE_SERIES:
        s = knoc_series_for_jodi(demand, key, value_col="value_kbd")
        if not s.empty:
            knoc_cmp_parts.append(s.set_index("date")["value_kbd"])
    knoc_compare_total = (
        pd.concat(knoc_cmp_parts, axis=1)
        .sum(axis=1)
        .reset_index()
        .rename(columns={0: "knoc_kbd"})
    )

    # --- JODI totals ---
    jodi_codes = [spec.jodi_energy_product for spec in JODI_COMPARE_SERIES.values()]
    jodi_compare_total = (
        jodi_kr[jodi_kr["energy_product"].isin(jodi_codes)]
        .groupby("date", as_index=False)["obs_value"]
        .sum()
        .rename(columns={"obs_value": "jodi_kbd"})
    )
    jodi_totprods = (
        jodi_kr[jodi_kr["energy_product"] == "TOTPRODS"][["date", "obs_value"]]
        .rename(columns={"obs_value": "jodi_kbd"})
    )

    # --- Plot 1: comparable (§7 scope) ---
    cmp = knoc_compare_total.merge(jodi_compare_total, on="date", how="inner")
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(
        x=cmp["date"], y=cmp["knoc_kbd"], name="KNOC (§7 product set)", line=dict(color="#1f77b4")
    ))
    fig1.add_trace(go.Scatter(
        x=cmp["date"], y=cmp["jodi_kbd"], name="JODI (§7 product set)", line=dict(color="#ff7f0e")
    ))
    fig1.update_layout(
        title="Total demand — KNOC vs JODI (same six products as §7, kbd)",
        yaxis_title="kbd", template="plotly_white", hovermode="x unified", height=420,
    )
    fig1.show()

    # Gap stats (last 24 months)
    cutoff = cmp["date"].max() - pd.DateOffset(months=23)
    m = cmp[cmp["date"] >= cutoff]
    gap = (m["knoc_kbd"] - m["jodi_kbd"]).abs().mean()
    print(f"Comparable total — mean |gap| (24m): {gap:,.1f} kbd ({gap / m['jodi_kbd'].mean() * 100:.1f}% of JODI)")

    # --- Plot 2: broad (TOTPRODS vs KNOC all products) ---
    broad = knoc_all_total.merge(jodi_totprods, on="date", how="inner")
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=broad["date"], y=broad["knoc_all_kbd"], name="KNOC (all parquet products)", line=dict(color="#1f77b4")
    ))
    fig2.add_trace(go.Scatter(
        x=broad["date"], y=broad["jodi_kbd"], name="JODI TOTPRODS", line=dict(color="#ff7f0e")
    ))
    fig2.update_layout(
        title="Broad total — KNOC (all products) vs JODI TOTPRODS (kbd)",
        yaxis_title="kbd", template="plotly_white", hovermode="x unified", height=420,
    )
    fig2.show()

    m2 = broad[broad["date"] >= cutoff]
    gap2 = (m2["knoc_all_kbd"] - m2["jodi_kbd"]).abs().mean()
    print(f"Broad total — mean |gap| (24m): {gap2:,.1f} kbd ({gap2 / m2['jodi_kbd'].mean() * 100:.1f}% of JODI)")
    print("Note: KNOC still excludes some JODI buckets (e.g. ONONSPEC); naphtha is in KNOC all but not in headline.")

Comparable total — mean |gap| (24m): 163.9 kbd (6.4% of JODI)


Broad total — mean |gap| (24m): 260.7 kbd (9.3% of JODI)
Note: KNOC still excludes some JODI buckets (e.g. ONONSPEC); naphtha is in KNOC all but not in headline.


In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=knoc_compare_total["date"], y=knoc_compare_total["knoc_kbd"],
    name="KNOC (§7 product set)",
))
fig.add_trace(go.Scatter(
    x=jodi_compare_total["date"], y=jodi_compare_total["jodi_kbd"],
    name="JODI (§7 product set)",
))
# KNOC shows through 2026-04; JODI through 2026-03

## 9. Jet fuel vs Kayrros nowcaster

Flight-based consumption (`kayros/jet_fuel`) vs KNOC **jet_fuel**.


In [11]:
import os

KAYROS_ROOT = PROJECT_ROOT.parent / "kayros" / "jet_fuel"
DB_PATH = KAYROS_ROOT / "data" / "jet_fuel.duckdb"
if not DB_PATH.exists():
    print(f"[skip] Kayrros DB not found at {DB_PATH}")
else:
    if str(KAYROS_ROOT) not in sys.path:
        sys.path.insert(0, str(KAYROS_ROOT))
    os.environ.setdefault("JET_FUEL_DB_PATH", str(DB_PATH))
    from src.export import get_consumption

    jet = demand[demand["product_native"] == "jet_fuel"][["date", "value_kbd"]].sort_values("date")
    now_raw = get_consumption(
        scope_type="country",
        scope="Korea (the Republic of)",
        freq="monthly",
        metric="avg_kbd",
        drop_incomplete=True,
    )
    now = now_raw.rename(columns={"period_start": "date", "value": "value_kbd"})
    plot = pd.concat([
        jet.assign(label="KNOC — jet_fuel"),
        now[["date", "value_kbd"]].assign(label="Kayrros — flights"),
    ], ignore_index=True)
    fig = px.line(plot, x="date", y="value_kbd", color="label", title="Korea jet fuel — KNOC vs Kayrros (kbd)")
    fig.update_layout(height=440, template="plotly_white", hovermode="x unified")
    fig.show()


## 10. Closing stocks (CLOSTLV)

Petronet *석유제품재고* — month-end inventories in **mbbl** (native kb ÷ 1000).

Baseline for the post–late-Feb 2026 window: **Feb 2026** closing level. MoM change = stock draw (negative) or build (positive).

In [12]:
if stocks.empty:
    print("[skip] No CLOSTLV rows — run: python scripts/update_korea.py --no-download --reparse-all")
else:
    WAR_BASELINE = pd.Timestamp("2026-02-01")
    stock_products = list(CHART_PRODUCTS)

    stk = stocks[stocks["product_native"].isin(stock_products)].copy()
    stk = stk.sort_values(["product_native", "date"])
    stk["delta_kb"] = stk.groupby("product_native")["value"].diff()
    stk["delta_mbbl"] = stk["delta_kb"] / 1000.0

    baseline = (
        stk[stk["date"] == WAR_BASELINE]
        .set_index("product_native")["value_mbbl"]
    )
    latest_date = stk["date"].max()
    latest = stk[stk["date"] == latest_date].set_index("product_native")["value_mbbl"]

    chg = (latest - baseline).rename("change_mbbl_since_feb2026")
    summary = pd.DataFrame({"feb_2026_mbbl": baseline, "latest_mbbl": latest}).join(chg)
    summary["latest_month"] = latest_date.strftime("%Y-%m")
    display(summary.sort_values("change_mbbl_since_feb2026").round(3))

    total = stocks.groupby("date", as_index=False)["value"].sum()
    total["value_mbbl"] = total["value"] / 1000.0
    total = total.sort_values("date")
    total["delta_mbbl"] = total["value_mbbl"].diff()

    recent = total[total["date"] >= "2025-10-01"]
    fig = px.line(
        recent,
        x="date",
        y="value_mbbl",
        title="Korea total product stocks (mbbl)",
        markers=True,
    )
    fig.add_shape(
        type="line",
        x0=WAR_BASELINE,
        x1=WAR_BASELINE,
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color="gray", width=1, dash="dash"),
    )
    fig.add_annotation(
        x=WAR_BASELINE,
        y=1.02,
        xref="x",
        yref="paper",
        text="Feb 2026 baseline",
        showarrow=False,
        font=dict(color="gray", size=11),
    )
    fig.update_layout(height=420, template="plotly_white", yaxis_title="mbbl")
    fig.show()

    mom = stk[stk["date"] >= "2025-10-01"].copy()
    fig2 = px.bar(
        mom,
        x="date",
        y="delta_mbbl",
        color="product_native",
        barmode="relative",
        title="MoM stock change by product (mbbl)",
        labels={"product_native": "product"},
    )
    fig2.update_layout(height=460, template="plotly_white")
    fig2.show()

,feb_2026_mbbl,latest_mbbl,change_mbbl_since_feb2026,latest_month
product_native,,,,
lpg,4.496,3.476,-1.020,2026-05
light_heavy_oil,0.158,0.154,-0.004,2026-05
heavy_oil,0.008,0.004,-0.004,2026-05
byproduct_fuel_oil,0.186,0.183,-0.003,2026-05
bunker_c,8.917,8.930,0.013,2026-05
kerosene,2.131,2.265,0.134,2026-05
gasoline,5.096,5.873,0.777,2026-05
jet_fuel,4.069,5.096,1.027,2026-05
diesel,10.844,13.252,2.408,2026-05


## 11. KNOC vs JODI (CLOSTLV, kb)

Side-by-side **closing stocks** from Petronet (*석유제품재고*) vs JODI secondary `CLOSTLV` for Korea (`KR`). Both series are in **thousand barrels** (KNOC native `kb` = JODI `KBBL`).

Same product panels as §8: fuel-oil components rolled up vs JODI `RESFUEL`. JODI also publishes `TOTPRODS` for a headline total comparison.

In [13]:
from analytics import cross_source_comparison_chart

JODI_PARQUET = PROJECT_ROOT / "data" / "processed" / "jodi" / "jodi_secondary.parquet"
if stocks.empty:
    print("[skip] No CLOSTLV rows in korea_knoc.parquet")
elif not JODI_PARQUET.exists():
    print(f"[skip] JODI parquet not found at {JODI_PARQUET}")
    print("       Run: python scripts/update_jodi.py")
else:
    jodi = pd.read_parquet(JODI_PARQUET)
    jodi["date"] = pd.to_datetime(jodi["date"])

    knoc_parts = []
    jodi_codes = []
    for key, spec in JODI_COMPARE_SERIES.items():
        kn = knoc_series_for_jodi(stocks, key, value_col="value")
        if kn.empty:
            continue
        kn = kn.groupby("date", as_index=False)["value"].sum()
        kn["panel"] = spec.panel
        knoc_parts.append(kn.rename(columns={"value": "value_kb"})[["date", "panel", "value_kb"]])
        jodi_codes.append(spec.jodi_energy_product)

    knoc_panel = pd.concat(knoc_parts, ignore_index=True)

    jodi_kr = jodi[
        (jodi["ref_area"] == "KR")
        & (jodi["flow_breakdown"] == "CLOSTLV")
        & (jodi["unit_measure"] == "KBBL")
        & (jodi["energy_product"].isin(jodi_codes))
    ].copy()
    code_to_panel = {
        spec.jodi_energy_product: spec.panel for spec in JODI_COMPARE_SERIES.values()
    }
    jodi_kr["panel"] = jodi_kr["energy_product"].map(code_to_panel)
    jodi_kr["value_kb"] = jodi_kr["obs_value"]

    _order = ["Diesel", "Fuel oil", "Gasoline", "Kerosene", "Jet fuel", "LPG", "Naphtha"]
    panels = [p for p in _order if p in knoc_panel["panel"].unique()]

    fig = cross_source_comparison_chart(
        df_a=knoc_panel,
        df_b=jodi_kr,
        products=panels,
        product_col_a="panel",
        product_col_b="panel",
        value_col_a="value_kb",
        value_col_b="value_kb",
        label_a="KNOC (Petronet)",
        label_b="JODI",
        title="Korea CLOSTLV — KNOC vs JODI (kb = KBBL)",
        units_label="kb",
        cols=2,
        panel_height=280,
    )
    fig.show()

    cutoff_24 = knoc_panel["date"].max() - pd.DateOffset(months=23)
    print("\nMean |gap| over last 24 months (kb):")
    for panel in panels:
        k = knoc_panel.loc[knoc_panel["panel"] == panel].set_index("date")["value_kb"]
        j = jodi_kr.loc[jodi_kr["panel"] == panel].set_index("date")["value_kb"]
        merged = pd.concat([k, j], axis=1, keys=["knoc", "jodi"]).dropna()
        merged = merged.loc[merged.index >= cutoff_24]
        if merged.empty:
            continue
        gap = (merged["knoc"] - merged["jodi"]).abs().mean()
        pct = gap / merged["jodi"].abs().mean() * 100
        print(f"  {panel:12s}  mean|gap| = {gap:>8,.0f} kb  ({pct:5.1f}% of JODI level)")

    # --- Headline totals (same product set as panels above) ---
    knoc_compare_total = (
        knoc_panel.groupby("date", as_index=False)["value_kb"].sum()
        .rename(columns={"value_kb": "knoc_kb"})
    )
    jodi_compare_total = (
        jodi_kr.groupby("date", as_index=False)["value_kb"].sum()
        .rename(columns={"value_kb": "jodi_kb"})
    )
    knoc_all_total = (
        stocks.groupby("date", as_index=False)["value"].sum()
        .rename(columns={"value": "knoc_kb"})
    )
    jodi_totprods = (
        jodi[
            (jodi["ref_area"] == "KR")
            & (jodi["flow_breakdown"] == "CLOSTLV")
            & (jodi["unit_measure"] == "KBBL")
            & (jodi["energy_product"] == "TOTPRODS")
        ][["date", "obs_value"]]
        .rename(columns={"obs_value": "jodi_kb"})
    )

    cmp = knoc_compare_total.merge(jodi_compare_total, on="date", how="inner")
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=cmp["date"], y=cmp["knoc_kb"], name="KNOC (§10 product set)", line=dict(color="#1f77b4")))
    fig2.add_trace(go.Scatter(x=cmp["date"], y=cmp["jodi_kb"], name="JODI (§10 product set)", line=dict(color="#ff7f0e")))
    fig2.update_layout(
        title="Total closing stocks — KNOC vs JODI (same products as panels, kb)",
        height=420,
        template="plotly_white",
        yaxis_title="kb",
        hovermode="x unified",
    )
    fig2.show()

    broad = knoc_all_total.merge(jodi_totprods, on="date", how="inner")
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=broad["date"], y=broad["knoc_kb"], name="KNOC (all products)", line=dict(color="#1f77b4")))
    fig3.add_trace(go.Scatter(x=broad["date"], y=broad["jodi_kb"], name="JODI TOTPRODS", line=dict(color="#ff7f0e")))
    fig3.update_layout(
        title="Broad total — KNOC (all Petronet products) vs JODI TOTPRODS (kb)",
        height=420,
        template="plotly_white",
        yaxis_title="kb",
        hovermode="x unified",
    )
    fig3.show()

    war_start = pd.Timestamp("2026-02-01")
    recent = cmp[cmp["date"] >= war_start].copy()
    if not recent.empty:
        recent["gap_kb"] = recent["knoc_kb"] - recent["jodi_kb"]
        print(f"\nSince {war_start.date()} (comparable product total, kb):")
        print(f"  Latest month     : {recent['date'].max().strftime('%Y-%m')}")
        print(f"  KNOC             : {recent.iloc[-1]['knoc_kb']:,.0f} kb")
        print(f"  JODI             : {recent.iloc[-1]['jodi_kb']:,.0f} kb")
        print(f"  KNOC − JODI      : {recent.iloc[-1]['gap_kb']:+,.0f} kb")


Mean |gap| over last 24 months (kb):
  Diesel        mean|gap| =    5,408 kb  ( 32.5% of JODI level)
  Fuel oil      mean|gap| =       76 kb  (  0.8% of JODI level)
  Gasoline      mean|gap| =    2,627 kb  ( 31.0% of JODI level)
  Kerosene      mean|gap| =    1,565 kb  ( 41.7% of JODI level)
  Jet fuel      mean|gap| =       65 kb  (  1.3% of JODI level)
  LPG           mean|gap| =    5,161 kb  ( 50.1% of JODI level)
  Naphtha       mean|gap| =       38 kb  (  0.3% of JODI level)



Since 2026-02-01 (comparable product total, kb):
  Latest month     : 2026-04
  KNOC             : 53,771 kb
  JODI             : 67,893 kb
  KNOC − JODI      : -14,122 kb
